In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
# 1. 데이터 로드 (Kaggle SECOM 데이터셋)
# secom.csv 에 590개의 센서 데이터 있음
data = pd.read_csv('uci-secom.csv')

In [4]:
# 결측치 비율이 40%를 넘는 열 제거
null_cols = data.columns[data.isnull().mean() > 0.4]
data.drop(columns=null_cols, inplace=True)

In [5]:
# 남은 결측치는 중앙값(Median)으로 단순 보정 수행
numeric_cols = data.select_dtypes(include='number').columns
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())

In [6]:
# 피처와 라벨 분리 (라벨 값 중 1은 불량, -1은 정상을 의미)
X = data.drop(columns=['Time', 'Pass/Fail'])
y = data['Pass/Fail']
# XGBoost는 0과 1의 타겟 라벨을 선호하므로 -1을 0으로 변환
y = y.replace(-1, 0)

In [7]:
# 2. 분산 임계치 필터링 (Variance Threshold)
# 값이 거의 일정하여 분석 가치가 없는 센서 제거 (분산이 0.05 이하인 열 제거)
selector = VarianceThreshold(threshold=0.05)
X_selected = selector.fit_transform(X)
selected_features = X.columns[selector.get_support()]
X = pd.DataFrame(X_selected, columns=selected_features)

In [8]:
# 3. 학습/테스트 데이터셋 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
# 4. 극단적 불균형 해결을 위한 SMOTE 오버샘플링 적용
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)


/Users/minah/Desktop/포트폴리오/semiconductor-yield-prediction/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [10]:
# 5. 특성 스케일링 (정규화)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)


In [11]:
# 6. XGBoost 하이퍼파라미터 그리드 서치 최적화
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'scale_pos_weight': [1, 5, 10]  # 불량 가중치
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='recall',  # 불량 검출력(Recall) 기준 최적화
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train_res)
best_model = grid_search.best_estimator_

Fitting 3 folds for each of 81 candidates, totalling 243 fits


In [12]:
# 7. 예측 수행 및 결과 레포트 출력
y_pred = best_model.predict(X_test_scaled)
print("--- Best Hyperparameters ---")
print(grid_search.best_params_)
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

--- Best Hyperparameters ---
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50, 'scale_pos_weight': 5}

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.02      0.03       293
           1       0.07      1.00      0.13        21

    accuracy                           0.08       314
   macro avg       0.53      0.51      0.08       314
weighted avg       0.94      0.08      0.04       314

